# ✅ Auth + Setup

## 🔐 **Authenticate to Google Cloud within Colab**

Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

In [2]:
from google.colab import auth

auth.authenticate_user()

## 💻 **Install Code Dependencies**
It is recommended to use the Connector alongside a library that can create connection pools, such as [SQLAlchemy](https://www.sqlalchemy.org/).
This will allow for connections to remain open and be reused, reducing connection overhead and the number of connections needed

Let's `pip install` the [Cloud SQL Python Connector](https://github.com/GoogleCloudPlatform/cloud-sql-python-connector) as well as [SQLAlchemy](https://www.sqlalchemy.org/), using the below command.

In [3]:
# install dependencies
import sys
!{sys.executable} -m pip install cloud-sql-python-connector["pymysql"] SQLAlchemy==2.0.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 kB 18.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 2.0.30
    Uninstalling SQLAlchemy-2.0.30:
      Successfully uninstalled SQLAlchemy-2.0.30
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.27.0
    Uninstalling google-auth-2.27.0:
      Successfully uninstalled google-auth-2.27.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.27.0, but you have google-auth 2.30.0 which is incompatible.


In [4]:
import google.auth
import pandas as pd

from google.cloud.sql.connector import Connector
from google.auth.transport.requests import Request
from sqlalchemy import create_engine, Table, Column, Integer, VARCHAR, ForeignKey, String, MetaData, text, select
from sqlalchemy.orm import Session

## 🐬 **Connect to a MySQL Instance**
We are now ready to connect to a MySQL instance using the Cloud SQL Python Connector! 🐍 ⭐ ☁


In [5]:
# initialize parameters
INSTANCE_CONNECTION_NAME = "inlaid-woods-388716:us-west1:ilkmaar" # i.e demo-project:us-central1:demo-instance
print(f"Your instance connection name is: {INSTANCE_CONNECTION_NAME}")

# IAM database user parameter (IAM user's email before the "@" sign, mysql truncates usernames)
# ex. IAM user with email "demo-user@test.com" would have database username "demo-user"
# grant Cloud SQL Client role to authenticated user
current_user = !gcloud auth list --filter=status:ACTIVE --format="value(account)"

IAM_USER = current_user[0].split("@")[0]
DB_NAME = "gameplay-data"

Your instance connection name is: inlaid-woods-388716:us-west1:ilkmaar


### ✅ **Connect to Database**
To connect to Cloud SQL using the connector, initialize a `Connector` object and call its `connect` method with the proper input parameters.

In [6]:
# initialize connector
connector = Connector()

# getconn now using IAM user and requiring no password with IAM Auth enabled
def getconn():
    conn = connector.connect(
      INSTANCE_CONNECTION_NAME,
      "pymysql",
      user=IAM_USER,
      db=DB_NAME,
      enable_iam_auth=True
    )
    return conn

# create connection pool
engine = create_engine(
    "mysql+pymysql://",
    creator=getconn,
)

def query_db(query_str):
    with engine.connect() as conn:
        return pd.read_sql_query(text(query_str), conn)

### Test Connection

This fails if the user does not have access to that database yet.

To fix:
  - log into the MySQL server on the Google Cloud Shell as root user
  - MySQL > GRANT ALL PRIVILEGES on `gameplay-data`.* to "user"@'%'

In [7]:
# connect to connection pool
with engine.connect() as db_conn:
    # get current datetime from database
    results = db_conn.execute(text("SELECT NOW()")).fetchone()

    # output time
    print("Current time: ", results[0])

ERROR:google.cloud.sql.connector.instance:['inlaid-woods-388716:us-west1:ilkmaar']: An error occurred while performing refresh. Scheduling another refresh attempt immediately
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/google/cloud/sql/connector/instance.py", line 188, in _refresh_task
    refresh_data = await refresh_task
  File "/usr/local/lib/python3.10/dist-packages/google/cloud/sql/connector/instance.py", line 131, in _perform_refresh
    connection_info = await self._client.get_connection_info(
  File "/usr/local/lib/python3.10/dist-packages/google/cloud/sql/connector/client.py", line 265, in get_connection_info
    metadata = await metadata_task
  File "/usr/local/lib/python3.10/dist-packages/google/cloud/sql/connector/client.py", line 127, in _get_metadata
    resp = await self._client.get(url, headers=headers, raise_for_status=True)
  File "/usr/local/lib/python3.10/dist-packages/aiohttp/client.py", line 696, in _request
    resp.raise_fo

ClientResponseError: 404, message='Not Found', url=URL('https://sqladmin.googleapis.com/sql/v1beta4/projects/inlaid-woods-388716/instances/ilkmaar/connectSettings')

## ❓Queries Setup

In [ ]:
query_store = {}

def ask(question):
    for report in query_store.values():
        if report['question'] == question:
            result = query_db(report['query'])
            return result
    return "Sorry, I don't have a query for that question."

def show(report_name):
    if report_name in query_store:
        result = query_db(query_store[report_name]['query'])
        return result
    else:
        return "Sorry, I don't have a query for that report."

players = query_db("SELECT * from players")
creatures = query_db("SELECT * from creatures")
location_groups = query_db("SELECT * from location_groups")
recipes = query_db("SELECT recipe_name from recipes")

def visitor_log_query(location_group):
    q = f"""
        SELECT
            creature_sightings.creature_sighting_time,
            creatures.creature_name
        FROM
            creature_sightings
        JOIN
            creatures on creature_sightings.creature_id = creatures.creature_id
        JOIN
            location_groups on creature_sightings.location_group_id = location_groups.location_group_id
        WHERE
            location_group_name like '{location_group}'
        ORDER BY time DESC
        LIMIT 10
    """
    return q

def recipe_book_query(recipe_name):
    q = f"""SELECT recipe_name, resource_type, resource_type_faction
        FROM recipes
        JOIN recipe_ingredient_resource_types ON recipes.recipe_id = recipe_ingredient_resource_types.recipe_id
        JOIN resource_types ON recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id
        WHERE recipes.recipe_name like '{recipe_name}'
    """
    return q

def cauldron_query():
    q = f"""
        SELECT
            DISTINCT recipes.recipe_name, recipes.recipe_temp, recipes.recipe_difficulty
        FROM
            recipes
        JOIN
            recipe_ingredient_resource_types on recipes.recipe_id = recipe_ingredient_resource_types.recipe_id
    """
    return q

def crafting_table_query(player_name):
    q = f"""
        SELECT
            crafting_event_time,
            recipe_name
        FROM
            crafting_events
        JOIN
            recipes on crafting_events.recipe_id = recipes.recipe_id
        JOIN
            collections on crafting_events.crafting_table_collection_id = collections.collection_id
        JOIN
            player_collection_access on collections.collection_id = player_collection_access.collection_id
        JOIN
            players on player_collection_access.player_id = players.player_id
        WHERE
            players.player_name like '{player_name}'
        ORDER BY time DESC
        LIMIT 10
    """
    return q

def trash_bin_query(player_name):
    q = f"""
        SELECT
            resource_transfer_time,
            resource_types.resource_type,
            resource_transfers.resource_id
        FROM
            resource_transfers
        JOIN
            collections on resource_transfers.destination_collection_id = collections.collection_id
        JOIN
            player_collection_access on collections.collection_id = player_collection_access.collection_id
        JOIN
            players on player_collection_access.player_id = players.player_id
        JOIN
            resources on resource_transfers.resource_id = resources.resource_id
        JOIN
            resource_types on resources.resource_type_id = resource_types.resource_type_id
        WHERE
            players.player_name like '{player_name}'
        AND
            collections.collection_type = 'crafting_table'
        ORDER BY resource_transfers.resource_transfer_time DESC
        LIMIT 30
    """
    return q

def creature_interactions_query(creature_name):
    q = f"""
        SELECT interaction_event_time, location_groups.location_group_name, locations.location_x, locations.location_y, players.player_name
        FROM
            interaction_events
        JOIN
            creatures on interaction_events.creature_id = creatures.creature_id
        JOIN
            players on interaction_events.player_id = players.player_id
        JOIN
            locations on interaction_events.location_id = locations.location_id
        JOIN
            location_groups on locations.location_group_id = location_groups.location_group_id
        WHERE
            creature_name like '{creature_name}'
    """
    return q

def player_foraging_query(player_name):
    q = f"""
          SELECT
              resource_transfers.resource_transfer_time,
              resource_transfers.resource_id,
              resource_types.resource_type as resource,
              source_collections.collection_name AS 'map area'
          FROM
              resource_transfers
          JOIN
              resources ON resource_transfers.resource_id = resources.resource_id
          JOIN
              resource_types ON resources.resource_type_id = resource_types.resource_type_id
          JOIN
              collections AS source_collections ON resource_transfers.source_collection_id = source_collections.collection_id
          JOIN
              collections AS destination_collections ON resource_transfers.destination_collection_id = destination_collections.collection_id
          JOIN
              player_collection_access ON destination_collections.collection_id = player_collection_access.collection_id
          JOIN
              players ON player_collection_access.player_id = players.player_id
          WHERE
              players.player_name like '{player_name}'
          AND
              destination_collections.collection_type = 'inventory'
          ORDER BY
              resource_transfers.resource_transfer_time DESC
      """
    return q

def player_gifting_query(player_name):
    q = f"""
        SELECT gifting_events.gifting_event_time, creatures.creature_name, recipes.recipe_name
        FROM
            gifting_events
        LEFT JOIN
            interaction_events on gifting_events.gifting_event_id = interaction_events.gifting_event_id
        JOIN
            creatures on interaction_events.creature_id = creatures.creature_id
        JOIN
            players on interaction_events.player_id = players.player_id
        JOIN
            item_transfers on gifting_events.item_transfer_id = item_transfers.item_transfer_id
        JOIN
            items on item_transfers.item_id = items.item_id
        JOIN
            recipes on items.recipe_id = recipes.recipe_id
        WHERE
            player_name like '{player_name}'
    """
    return q

def creature_gifts_query(creature_name):
    q = f"""
        SELECT gifting_events.gifting_event_time, players.player_name, recipes.recipe_category
        FROM
            gifting_events
        LEFT JOIN
            interaction_events on gifting_events.gifting_event_id = interaction_events.gifting_event_id
        JOIN
            creatures on interaction_events.creature_id = creatures.creature_id
        JOIN
            players on interaction_events.player_id = players.player_id
        JOIN
            item_transfers on gifting_events.item_transfer_id = item_transfers.item_transfer_id
        JOIN
            items on item_transfers.item_id = items.item_id
        JOIN
            recipes on items.recipe_id = recipes.recipe_id
        WHERE
            creature_name like '{creature_name}'
    """
    return q

for _ , player in players.iterrows():
    player_name = player['player_name']
    query_store[f"{player_name}'s Crafting Table Log"] = {
        'question': f"Which recipes has {player_name} crafted recently?",
        'query': crafting_table_query(player_name)
    }

for _ , player in players.iterrows():
    player_name = player['player_name']
    query_store[f"{player_name}'s Gifting Log"] = {
        'question': f"Which items has {player_name} gifted recently?",
        'query': player_gifting_query(player_name)
    }

for _ , player in players.iterrows():
    player_name = player['player_name']
    query_store[f"{player_name}'s Foraging Log"] = {
        'question': f"Which resources has {player_name} foraged recently?",
        'query': player_foraging_query(player_name)
    }

for _ , player in players.iterrows():
    player_name = player['player_name']
    query_store[f"{player_name}'s Trash Bin"] = {
        'question': f"Which items has {player_name} consumed recently?",
        'query': trash_bin_query(player_name)
    }

for _ , creature in creatures.iterrows():
    creature_name = creature['creature_name']
    query_store[f"{creature_name}'s Gift Memories"] = {
        'question': f"What gifts has {creature_name} gotten recently?",
        'query': creature_gifts_query(creature_name)
    }

for _ , creature in creatures.iterrows():
    creature_name = creature['creature_name']
    query_store[f"{creature_name}'s Location Memories"] = {
        'question': f"Who has {creature_name} interacted with recently?",
        'query': creature_interactions_query(creature_name)
    }

for _ , location_group in location_groups.iterrows():
    location_group_name = location_group['location_group_name']
    query_store[f"{location_group_name} Visitor Log"] = {
        'question': f"Who has been seen in {location_group_name} recently?",
        'query': visitor_log_query(location_group_name)
    }

for _ , recipe in recipes.iterrows():
    recipe_name = recipe['recipe_name']
    query_store[f"{recipe_name} Recipe"] = {
        'question': f"What ingredients do I need to cook {recipe_name}?",
        'query': recipe_book_query(recipe_name)
    }

query_store['Cauldron Instructions'] = {
    'question': f"What temperature do I cook recipes at?",
    'query': cauldron_query()
}

query_store['Triage Critical Condition List'] = {
    'question': f"Who are the sickest creatures?",
    'query': "SELECT creature_name, creature_faction, creature_health FROM creatures ORDER BY creature_health ASC LIMIT 10"
}

In [ ]:
!{sys.executable} -m pip install plotly

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

def show_as_table(title):
    query = query_store[title]['query']
    df = query_db(query)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df.columns),
                    fill_color='paleturquoise',
                    align='left'),
        cells=dict(values=[df[col] for col in df.columns],
                  fill_color='lavender',
                  align='left'))
    ])
    fig.show()

def show_as_bar_chart(title, categorical_column_for_x, numeric_column_for_y):
    query = query_store[title]['query']
    df = query_db(query)

    color_dict = {
        'Shadow': 'darkblue',
        'Light': 'yellow',
        'Growth': 'green',
        'Stability': 'grey'
    }

    # Create a bar chart
    fig = go.Figure(data=[
        go.Bar(
            name='Column 1',
            x=df[categorical_column_for_x], # categorical type
            y=df[numeric_column_for_y], # numeric type
            marker_color=[color_dict[i] for i in df[categorical_column_for_x]]
        )
    ])
    fig.update_layout(barmode='group')
    fig.show()

def show_as_plot(title, column_x, column_y):
    # Create a scatter plot
    query = query_store[title]['query']
    df = query_db(query)

    fig = px.scatter(df, x=column_x, y=column_y)

    fig.show()

def show_as_map(title, x, y, color_column):
    query = query_store[title]['query']
    df = query_db(query)

    # Create a scatter plot
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df[x],
            y=df[y],
            mode='markers',
            marker=dict(color=color_column)
        )
    )

    fig.update_layout(
        autosize=False,
        width=651,
        height=605,
        images=[dict(
            source="https://i.imgur.com/MH1kzqg.png",  # replace with your own image URL
            xref="x",
            yref="y",
            x=-100,
            y=100,
            sizex=200,
            sizey=200,
            sizing="stretch",
            opacity=0.5,
            layer="below")])

    # Set the range of x and y axis with padding
    fig.update_xaxes(range=[-100, 100])
    fig.update_yaxes(range=[-100, 100])

    fig.show()

# 🔍 An "Anomalous" Investigation

In [1]:
#
# I'm in the potions clinic. I look at the creatures in line.
# I look at the list that shows them by least healthy, so I know who to help most.
#

show("Triage Critical Condition List")

#
# Amethyst is the most sick.
#

NameError: name 'show' is not defined

In [ ]:
#
# Amethyst is really sick. Why?
#

#
# I read in the Newspaper that some creatures might get sick if they
# go to a contaminated location, or eat items that make them sick.
# I'll ask Amethyst where she remembers visiting.
# Maybe there will be a pattern with other sick creatures.
#

show("Amethyst 's Location Memories") # ignore the extra space in the name i forgot to sanitize input data

#
# She went to the Growth Tree most... but also went to other places, idk
#

In [ ]:
#
# Sometimes they're sick because they ate something poisonous?
# I'll ask Aurora what gifts she has been given to eat.
#

show("Amethyst 's Gift Memories")

#
# I see that Emiko gave her a treat.
# And Hannah gave her a potion just yesterday.
#

In [ ]:
#
# Hannah and Emiko gave me access to their gifting data. Let me check...
#

#show("Emiko's Gifting Log")
show("Hannah's Gifting Log")

#
# Ah, I see that Emiko gave Amethyst a "Golden Honeydew Fusion"
# And Hannah gave her a "Growthfire Leaf Fusion"...
#

In [ ]:
# Q: Does 'Growthfire Leaf Fusion' make Light types sick?
# I could look at the potions clinic log and see if other sick Light creatures had that.

# I remember reading in the Newspaper...
# that some types of recipes could make creatures sick depending on their faction...

# Let's see what ingredients are in the recipes Amethyst ate...

show('Growthfire Leaf Fusion Recipe') # Tranquil Moss (Growth) and Sugarleaf (Growth)
# show('Golden Honeydew Fusion Recipe')  # Stormfruit (Shadow) and Honeydew (Growth)

In [ ]:
# I see both of those are Growth type ingredients.
# So Maybe Growth type ingredients make Light type sick...
# I'd check the 'Golden Honeydew Fusion' recipe, but I don't have access yet!

# Maybe the other creatures in triage right now will also have eaten these ingredients?
# I'll go back to the Triage (and/or figure out where to find Iris) and ask her what she has been given.

show("Iris's Gift Memories")

In [ ]:
#show("Dalia's Gifting Log")
show("Hannah's Gifting Log")

# I see Dalia gave Iris an "Umbral Stonewood Token"
# and Hannah gave her a "Serene Hardwood Essence"... what even is that
# Lemme look at those recipes too

#show("Serene Hardwood Essence Recipe") # Ingredients: Stonewood (Stability), Hardwood (Growth)
show("Umbral Stonewood Token Recipe") # Ingredients: Stonewood (Stability), Shadowglass (Shadow)

# Kinda weird that creatures eat Stone, Hardwood and Glass but go off
# oh wait maybe that's bad for them
# oh wait she doesn't eat any Light type foods!
# maybe the other healthy creatures are actually just eating foods that are good for them
# ...how could I find healthiest creatures?

In [ ]:
# idk what kind of world object would maintain a list of healthiest creatures
# maybe we need a "Gold Star for Health Record" on the wall in the Potions Clinic...
# (so let's make one:)

query_store["Gold Star for Health Record"] = {
    'question': "Who are the healthiest creatures of them all?",
    'query': "SELECT creature_name, creature_health FROM creatures ORDER BY creature_health DESC LIMIT 10"
}

show("Gold Star for Health Record") # would be nice to attach these different queries to specific world objects...

In [ ]:
# Viridius! Who is that? I don't remember who "Viridius" is or what faction. Sounds Green

# You know what might be useful?
# A way to look up creature factions by name

query_store["Creature Faction Lookup Book"] = {
    'question': "What are all the creatures' factions?",
    'query': "SELECT creature_name, creature_faction, creature_color FROM creatures"
}

show("Creature Faction Lookup Book")

# OK I'm not reading all that
# Let's... 'flip to the page' on V

book = show("Creature Faction Lookup Book")
book[book['creature_name'] == "Viridius"]

In [ ]:
# OK! So Viridius is Growth type
# Let's see what he ate that made him so health

show("Viridius's Gift Memories") # (Emiko, Potion) (Giselle, Food) (Chiara, Potion)

#
# Can look at players' gifting logs again...
# show("Chiara's Gifting Log") # Tranquility Moss Tonic (P), Twilight Stormfruit Surprise (F), Stalwart Stew (P)

# Then at ingredients again...

# ...boy this sure would be easier if I just looked at the crafting Logs,
# which in an updated version should definitely tell me the resources used to craft an item
# But anyway...

#show("Twilight Stormfruit Surprise Recipe") #Food: (Shadow, Growth)
#show("Tranquility Moss Tonic Recipe") #Potion: (Light, Growth)
show("Stalwart Stew Recipe") #Potion: (Light, Growth)

# OK so this one always eats foods with one of their own type of ingredient in them...!

In [ ]:
# Actually I wonder if the ones in the middle range of health
# just keep eating both good and bad foods that balance out
# and maybe they eat more foods overall, they just balance
# like a food random walk

# This could/should just be medical records in the potions clinic but maybe u need access
# But maybe someone/somewhere in the game can have like all current records on something.
# e.g. a "Health Oracle"...

query_store["The Health Oracle"] = {
    'question': "What are all creatures' health levels?",
    'query': "SELECT creature_name, creature_faction, creature_health from creatures"
}

#show("The Health Oracle")

# Hmm we should have one for Mood, too
query_store["The Mood Oracle"] = {
    'question': "What are all creatures' mood levels?",
    'query': "SELECT creature_name, creature_faction, creature_mood from creatures ORDER BY creature_mood DESC"
}

show("The Mood Oracle")

# And why not a Top 10 like in the "Gold Star for Health Poster"
# Maybe something like this is a daily Newspaper 'article'...?
# "10 Healthiest Creatures: What do they eat!?"
# Could alternate with "10 Saddest Creatures: what makes them tick?"

query_store["Top 10 Happiest Creatures"] = {
    'question': "What are all creatures' mood levels?",
    'query': "SELECT creature_name, creature_faction, creature_mood from creatures ORDER BY creature_mood DESC LIMIT 10"
}

show("Top 10 Happiest Creatures")

# HA! The happiest creature is named "Gloomsong"...

In [ ]:
# OK let's see what makes creatures so happy...
# (this actually seems like a very good type of "onboarding" clue)

show("Gloomsong's Gift Memories") # see: (0, Jasmine, Gift); (6, Dalia, Treat)
# show("Radiance's Gift Memories") # see 4, one of each type

# Ah, so Gloomsong got a Gift and a Treat and is happiest of all...
# Gifts and Treats definitely would have a mood effect...
# But wait, Radiance got a Treat, Food, Potion AND a Gift and is *less* happy...!

# So maybe the KIND of gifts/treats matters

In [ ]:
# What did Jasmine and Dalia gift that Gloomsong liked so much?

# show("Jasmine's Gifting Log") # see: gave Gloomsong "Tranquilwood Harmony"
show("Dalia's Gifting Log") # see: gave Gloomsong "Golden Honeydew Fusion"

show("Tranquilwood Harmony Recipe") # (Nightshade, Shadow), (Shadowglass, Shadow)
# show("Golden Honeydew Fusion Recipe") # (Stormfruit, Shadow), (Honeydew, Growth)

# OK so this is REALLY tedious to keep looking up Gifts and THEN their recipes!
# What if there was some kind of way... to combine two information sources...

# So first let's make a more useful cookbook, which has faction type information too.
# (Maybe we unlock this by levelling up in Cooking or something.)

# Let's make a Super Useful Cook Book.
query_store["Super Useful Cook Book"] = {
    'question': "How can I cook everything?",
    'query': """
        SELECT recipes.recipe_id, recipes.recipe_name, resource_types.resource_type, resource_types.resource_type_faction
        FROM recipe_ingredient_resource_types
        JOIN recipes on recipe_ingredient_resource_types.recipe_id = recipes.recipe_id
        JOIN resource_types on recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id
    """
}

# Actually let's make an Extremely Useful Cook Book. This one has 'rarity' information, too.
query_store["Extremely Useful Cook Book"] = {
    'question': "How can I cook everything and also know details?",
    'query': """
        SELECT recipes.recipe_id, recipes.recipe_name, resource_types.resource_type, resource_types.resource_type_faction, resource_types.resource_type_rarity
        FROM recipe_ingredient_resource_types
        JOIN recipes on recipe_ingredient_resource_types.recipe_id = recipes.recipe_id
        JOIN resource_types on recipe_ingredient_resource_types.resource_type_id = resource_types_resource.type_id
    """
}

# No wait we could even make an God Mode Cook Book
query_store["God Mode Cook Book"] = {
    'question': "How can I know everything about recipes?",
    'query': """
        SELECT *
        FROM recipe_ingredient_resource_types
        JOIN recipes on recipe_ingredient_resource_types.recipe_id = recipes.recipe_id
        JOIN resource_types on recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id
    """
}

# OK perfect-- so say we have one source of data: A Cook Book.
# Now let's say we have another: "Gloomsong's Gift Memories", the one we were just looking at.

# What if we could... squish them together...
# so that the Gifting Log also had all the information from the Cook Book?

def squish(title1, title2):
    query1 = query_store[title1]['query']
    query2 = query_store[title2]['query']

    new_query = f"""
        SELECT *
        FROM ({query1}) as table1
        JOIN ({query2}) as table2
        ON table1.recipe_name = table2.recipe_name;
    """

    return query_db(new_query)

# Now instead of going around looking up recipes whenever we want to figure out what a creature ate,
# we can just "squish" our Cook Book onto it!
squish("Jasmine's Gifting Log", "Super Useful Cook Book")

In [ ]:
# But yuck, that's a lot of columns. And a bunch are redundant.
# This can be improved if in my query_store I store column names, standardize them,
# and build the query in a smarter way...

query_store["Super Useful Cook Book"]['columns'] = ['recipe_id', 'recipe_name', 'resource_type', 'resource_type_faction']
query_store["Extremely Useful Cook Book"]['columns'] = ['recipe_id', 'recipe_name', 'resource_type', 'resource_type_faction']
# Note this wouldn't work yet with my God Mode cookbook because of the SELECT *
query_store["Jasmine's Gifting Log"]['columns'] = ['gifting_event_time', 'creature_name', 'recipe_name']
query_store["Creature Faction Lookup Book"]['columns'] = ['creature_name', 'creature_faction', 'creature_color']
query_store["The Mood Oracle"]['columns'] = ['creature_faction', 'creature_name', 'creature_mood']

def squish(title1, title2):
    query1 = query_store[title1]['query']
    query2 = query_store[title2]['query']

    columns1 = set(query_store[title1]['columns'])
    columns2 = set(query_store[title2]['columns'])

    joinable_columns = list(columns1.intersection(columns2))
    columns_to_select = ", ".join(list(columns1.union(columns2)))

    if len(joinable_columns) == 1:
        join_column = joinable_columns[0]

        # Remove join_column from the sets and form the rest of the select statement
        columns1.discard(join_column)
        columns2.discard(join_column)
        columns_to_select = ", ".join(["table1." + col for col in columns1] + ["table2." + col for col in columns2])

        new_query = f"""
            SELECT table1.{join_column}, {columns_to_select}
            FROM ({query1}) as table1
            JOIN ({query2}) as table2
            ON table1.{join_column} = table2.{join_column}
        """
        return query_db(new_query)
    elif len(joinable_columns) > 1:
        return "Ambiguous. Need more info."
    else:
        return f"No can do squish. Columns1: {columns1}, Columns2: {columns2}"

# Excellent!
squish("Jasmine's Gifting Log", "Super Useful Cook Book")

In [ ]:
# Now if I was in the medical building (or anywhere) and I wanted to know what ingredients were used...
# I could just "squish" my Cook Book onto it.

# Note: squishing the opposite direction should work too-- squishing the Gifting Log onto the Cook Book.
# The distinction matters when one has partial information-- e.g. I don't have access to all recipes yet.

# Also note: this basically gives me a list of all ingredients the player has ever used in their gifts.
# If the creatures had actually remembered gift item (instead of just gift type) this would be another way
# to create a list of all ingredients the creature had consumed.

# Which in this specific fake investigation would be pretty useful!
# So instead what we could do...

# What if we looked at all gifting data at once (that we had access to, say)?
# Or, if all/most crafting is done on a "community crafting table"-- I could look at those records and squish my cook book onto it too.

query_store["All-Player Shared Gifting Log"] = {
    'question': "What has everyone gifted?",
    'columns': ['gifting_event_time', 'recipe_name', 'creature_name'], # should match individual gift logs...
    'query': """
        SELECT gifting_events.gifting_event_time, recipes.recipe_name, creatures.creature_name
        FROM gifting_events
        JOIN item_transfers on gifting_events.item_transfer_id = item_transfers.item_transfer_id
        JOIN items on item_transfers.item_id = items.item_id
        JOIN recipes on items.recipe_id = recipes.recipe_id
        JOIN creatures on gifting_events.creature_id = creatures.creature_id
    """
}

# OK now we have a log of all gifts:
show("All-Player Shared Gifting Log")

In [ ]:
# But we only want to look at Gloomsong. Or Amethyst.
book = show("All-Player Shared Gifting Log")
book[book['creature_name'] == 'Gloomsong']

# Now we have everything they've been gifted.

In [ ]:
# Let's squish it with our Cook Book
book = squish("All-Player Shared Gifting Log", "Super Useful Cook Book")

book[book['creature_name'] == 'Gloomsong'] # And filter down to only for the creature we're interested in...

# Now we have all ingredients Gloomsong has eaten!
# (I'm getting convinced here that tables + filtering + squishing will go a long way)

In [ ]:
# Ah but wait, wouldn't it be nice to be able to squish this *again* with my Creature Phone Book?
# Then I could do things like see creature Faction compared to what they ate.
# Which would normally be complicated but in this game could just be "squishing" 2 items together
# (Which could also re-use the inventory and crafting UI for combining 2 objects into a new one)

# So if I 'squished' Gifting Log + Cook Book with the Health Oracle (or any recent medical records) I could see...
# creature health AND everything they ate.
# That seems pretty useful!

# But we'd need a new and improved squish... that could be *chained*

# So I'll have 'squish' return the title of a new object that can be looked up in query_store
# and then I'll have to look up and run the final result query

def squish(title1, title2):
    q1 = query_store[title1]
    q2 = query_store[title2]

    query1 = q1['query']
    query2 = q2['query']

    # This is the cringiest code ever dont look
    columns1 = set(q1['columns'])
    columns2 = set(q2['columns'])
    columns1_only = columns1 - columns2
    columns2_only = columns2 - columns1
    shared_columns = list(columns1.intersection(columns2))
    unshared_columns = list(columns1.union(columns2))

    if len(shared_columns) >= 1:
        new_title = f"{title1}_squish_{title2}"

        # need to replace t1 and t2 now since they'll make the 'selects' ambiguous. ugh
        table1 = "`" + title1 + "`"
        table2 = "`" + title2 + "`"

        columns_to_select = ", ".join([f"{table1}." + col for col in columns1] + [f"{table2}." + col for col in columns2_only])
        join_condition = " AND ".join([f"{table1}.{col} = {table2}.{col}" for col in shared_columns])

        new_query = f"""
            SELECT {columns_to_select}
            FROM ({query1}) as {table1}
            JOIN ({query2}) as {table2}
            ON {join_condition}
        """

        query_store[new_title] = {
            'question': f"What do you get when you squish {title1} and {title2}?",
            'columns': shared_columns + list(columns1_only) + list(columns2_only),
            'query': new_query
        }
        return new_title
    else:
        return f"No can do squish. Columns1: {columns1}, Columns2: {columns2}"

# Let's see if this works.
# This represents squishing the Gifting Log and Cook Book...
# and THEN squishing the Creature Faction Lookup Book
#squish("All-Player Shared Gifting Log", "Super Useful Cook Book")


# YASSSSSSSS
title = squish(squish("All-Player Shared Gifting Log", "Super Useful Cook Book"), "Creature Faction Lookup Book")
query_db(query_store[title]['query'])

# It's... beautiful...
# tho would benefit from column ordering settings and naming etc, order gets lost in the 'set' I think

In [ ]:
# Let's keep going and squish on the Health Oracle too!

# Gotta give it some columns first
query_store["The Health Oracle"]['columns'] = ['creature_name', 'creature_faction', 'creature_health']

title = squish(squish(squish("All-Player Shared Gifting Log", "Super Useful Cook Book"), "Creature Faction Lookup Book"), "The Health Oracle")
query_db(query_store[title]['query'])

# OMG IT WORKS

In [ ]:
from sqlalchemy.sql.ddl import sort_tables_and_constraints

# Now we can see all ingredients that Gloomsong ate (and all the factions those belong to)
# book = query_db(query_store[title]['query'])
# book[book['creature'] == 'Gloomsong']

book[book['creature_faction'] == 'Shadow']

# OK. We have all the data we would need to figure out if creature_health depends on types of item consumed.
# BUT. I can't really do anything good with this
# Normally I'd just have a SQL query that grouped or ordered things, but this was created through 'squishes'
# I need a way to sort and group and stuff

# OK... what IF
# what if I can do what I did for 'squish' but again for a 'sort':

def sort(title, column):
    q = query_store[title]['query']
    columns = query_store[title]['columns']

    new_title = title + "_sortdesc_" + column

    new_query = f"""
        SELECT *
        FROM ({q}) as table_alias
        ORDER BY table_alias.{column} DESC
    """
    query_store[new_title] = {
        'question': f"What do you get when you sort {title} by '{column}'?",
        'columns': columns,
        'query': new_query
    }
    return new_title

# NICE.
# What this means is that we can manipulate tables like they're inventory objects.
# And use "tools" in our inventory to manipulate tables too.
# A "sorter tool" could be used on a table (just select which column) and the query that the object represents gets updated.
# So a DataInventoryObject has an associated query and a set of Squish events and Sorter/Group applications.
# They can be stored in a chain, and reversed with an "undo"

q = sort(sort(squish(squish(squish("All-Player Shared Gifting Log", "Super Useful Cook Book"), "Creature Faction Lookup Book"), "The Health Oracle")
, 'creature_faction'), 'resource_type_faction')
query_db(query_store[q]['query'])

In [ ]:
# Let's make one more tool:
# a 'group-averager'
# advanced tool. much upgrade

def group_averager(title, columns_to_group, column_to_average):
    q = query_store[title]['query']
    columns = query_store[title]['columns']

    new_title = title + "_group_" + columns_to_group[0] + "_avg_" + column_to_average
    columns_to_group_by = ", ".join(columns_to_group)
    all_columns = columns_to_group.append(column_to_average)

    new_query = f"""
        SELECT {columns_to_group_by}, AVG({column_to_average})
        FROM ({q}) as table_alias
        GROUP BY {columns_to_group_by}
        ORDER BY AVG({column_to_average}) DESC
    """
    query_store[new_title] = {
        'question': f"What do you get when you average {column_to_average} by '{columns_to_group}'?",
        'columns': all_columns,
        'query': new_query
    }
    return new_title

#### OMGGGGG YAYYYYYY

q = sort(sort(squish(squish(squish("All-Player Shared Gifting Log", "Super Useful Cook Book"), "Creature Faction Lookup Book"), "The Health Oracle")
, 'creature_faction'), 'resource_type_faction')
n = group_averager(q, ['creature_faction'], 'creature_health')
query_db(query_store[n]['query'])

### We got to "average faction health"
### from: crafting data, a cook book, a phone book, a health oracle, and an "averager" tool
### Is it insane? Yes
### Can we do data science by 'crafting' with really basic inventory objects? Yes

In [ ]:
# OK so A: it looks like mood for Shadow type most affected by gifts/treats with Shadow ingredients!
# That would make sense...
# Just like Growth health was most affected by Growth ingredients in the Foods and Potions

# What kind of weird query would convince us of this?
# We could
#   1) look at all Potions made with Shadow ingredients and see if Shadow creatures who got those are healthier
#   2) look at all Potions with TWO Shadow ingredients and see if Shadow creatures who got those are healthier
#   3) maybe the opposite is true, and Potions made with Light ingredients make creatures most unhappy?

# OK let's go back to
q = group_averager(squish(squish(squish("All-Player Shared Gifting Log", "Super Useful Cook Book"), "Creature Faction Lookup Book"), "The Health Oracle"), ['creature_faction', 'resource_type_faction'], 'creature_health')
r = query_db(query_store[q]['query'])
r[r['creature_faction'] == "Light"]

In [ ]:
# Let's see if I can figure out where those ingredients were from.
# Maybe they were contaminated.

# Hmmm did Hannah craft this or trade for it in the shop?


# show("Hannah's Crafting Table Log")

# Hmm I see she made it on Day 8. But otherwise nothing really new here.

#show("Hannah's Trash Bin")

# find on day 8: Tranquil Moss#2333
# find on day 8: Sugarleaf#2331

# How could I track down where those came from...?
# If she traded for them with another player I could look at the Shop Exchange log...

# But maybe she foraged them herself... and I don't have access to her foraging data...
# Ah, I can bet where they came from.

# Where have *I* found Sugarleaf and Tranquilmoss?
show("Hannah's Foraging Log")

# Hmm that doesn't seem helpful! Too much info.
# I'll just select the resources I'm interested in...
r = show("Hannah's Foraging Log")
r[r['resource_id'] == 2331]

# Stability Mines and Growth Tree...

In [ ]:
# WHERE HAS TRANQUIL MOSS BEEN

# World Data:
#
#   - Visitor Logs for each map area ("location group")
#   - Crafting Table Log for each player
#
# To get player or map area names:
#  query_db("SELECT * from players")['player_name'].tolist()
#  query_db("SELECT * from location_groups")['location_group_name'].tolist()
#  query_db("SELECT * from recipes")['recipe_name'].tolist()

# Look at a Visitor Log:
# show("Light Garden Visitor Log")

# Look at Cauldron instructions:
# show('Cauldron Instructions')

resource_type = "Tranquil Moss"  # Replace with the desired resource type

query_db(
    f""" SELECT
    resource_types.resource_type as resource,
    source_collections.collection_name AS source,
    COUNT(*) as count
FROM
    resource_transfers
JOIN
    resources ON resource_transfers.resource_id = resources.resource_id
JOIN
    resource_types ON resources.resource_type_id = resource_types.resource_type_id
JOIN
    collections AS source_collections ON resource_transfers.source_collection_id = source_collections.collection_id
JOIN
    collections AS destination_collections ON resource_transfers.destination_collection_id = destination_collections.collection_id
JOIN
    player_collection_access ON destination_collections.collection_id = player_collection_access.collection_id
JOIN
    players ON player_collection_access.player_id = players.player_id
WHERE
    destination_collections.collection_type = 'inventory'
AND
    resource_types.resource_type = '{resource_type}'
GROUP BY resource_types.resource_type, source_collections.collection_name
ORDER BY count DESC
"""
)

# ❓ Demo Data


## 🍎 resources

#### *How many resources have been taken from each area?*

In [ ]:
query_db("""
SELECT collection_name as 'Map Area', COUNT(*) as count
FROM resource_transfers
JOIN collections ON resource_transfers.source_collection_id = collections.collection_id
JOIN resources on resource_transfers.resource_id = resources.resource_id
JOIN resource_types on resources.resource_type_id = resource_types.resource_type_id
WHERE collection_type = 'map_area'
GROUP BY collection_name
ORDER BY count DESC
LIMIT 10
""")

NameError: ignored

#### *Which resource types have been used in crafting the most by players?*

In [ ]:
query_db(f"""
SELECT
    resource_types.resource_type_faction as 'type', COUNT(*) as 'total'
FROM
    resource_transfers
JOIN
    collections ON resource_transfers.destination_collection_id = collections.collection_id
JOIN
    resources ON resource_transfers.resource_id = resources.resource_id
JOIN
    resource_types ON resources.resource_type_id = resource_types.resource_type_id
WHERE
    collections.collection_type = 'crafting_table'
GROUP BY resource_types.resource_type_faction
ORDER BY total DESC;
""")

,type,total
0,Shadow,214
1,Light,175
2,Growth,174
3,Stability,140


#### *Where did Shadowberries that were picked go?*

In [ ]:
resource_name = "Shadowberry"

query_db(f"""
SELECT collections.collection_name, COUNT(*) as count
FROM resources
JOIN resource_types ON resources.resource_type_id = resource_types.resource_type_id
JOIN collections ON resources.collection_id = collections.collection_id
WHERE resource_types.resource_type = '{resource_name}'
GROUP BY collections.collection_name
ORDER BY COUNT(*) DESC;
""")

,collection_name,count
0,Bianca's Crafting Table,8
1,Jasmine's Crafting Table,7
2,Dalia's Crafting Table,6
3,Fatima's Crafting Table,5
4,Emiko's Crafting Table,5
5,Isabella's Crafting Table,5
6,Hannah's Crafting Table,4
7,Giselle's Crafting Table,3
8,Aisha's Crafting Table,2
9,Chiara's Crafting Table,1


## 🐱 creature interactions

#### *Where has Aurora been seen?*

In [ ]:
creature_name = "Aurora"

query_db(f"""
SELECT
    interaction_event_time as day,
    location_groups.location_group_name as 'map area'
FROM interaction_events
JOIN locations on interaction_events.location_id = locations.location_id
JOIN location_groups on locations.location_group_id = location_groups.location_group_id
JOIN creatures ON interaction_events.creature_id = creatures.creature_id
WHERE creatures.creature_name = '{creature_name}';
""")

,day,map area
0,1,Light Garden
1,1,Light Garden
2,3,Light Tree
3,5,Shadow Patch


#### *How many creatures have been seen in The Rock Garden?*

In [ ]:
area_name = "Rock Garden"

query = f"""
SELECT COUNT(DISTINCT interaction_events.creature_id) AS distinct_creature_count
FROM interaction_events
JOIN locations ON interaction_events.location_id = locations.location_id
JOIN location_groups ON locations.location_group_id = location_groups.location_group_id
WHERE location_groups.location_group_name = '{area_name}';
"""

query_db(query)

,distinct_creature_count
0,26


#### *How many times has each faction been interacted with?*

In [ ]:
query_db("""
SELECT creature_faction as 'faction', COUNT(interaction_event_id) as interactions
FROM interaction_events
JOIN creatures ON interaction_events.creature_id = creatures.creature_id
WHERE interaction_events.interaction_event_time BETWEEN '0' AND '10'
GROUP BY creature_faction
ORDER BY interactions DESC;
""")

,faction,interactions
0,Stability,91
1,Light,83
2,Shadow,81
3,Growth,77


#### *Which Creatures have been interacted with in The Rock Garden in the last 3 days?*

In [ ]:
area_name = "Rock Garden"
days = 3

query_db(f"""
SELECT
    DISTINCT creatures.creature_name,
    interaction_events.interaction_event_time as day
FROM interaction_events
JOIN locations ON interaction_events.location_id = locations.location_id
JOIN location_groups ON locations.location_group_id = location_groups.location_group_id
JOIN creatures ON interaction_events.creature_id = creatures.creature_id
WHERE location_groups.location_group_name = '{area_name}' AND interaction_events.interaction_event_time >= (SELECT MAX(interaction_event_time) - {days} FROM interaction_events);
""")

,creature_name,day
0,Shadowfire,6
1,Gloomsong,6
2,Twilightspire,6
3,Vanguard,6
4,Stalwart,6
5,Stonewall,6
6,Oakheart,9
7,Ambrose,9
8,Shadowthorn,9
9,Guardian,9


#### *How many creatures of each type have been seen on Light Island in the last 10 days?*

In [ ]:
island = "Light"
days = 10

query = f"""
SELECT creatures.creature_faction, COUNT(interaction_events.interaction_event_id) AS count
FROM interaction_events
JOIN locations ON interaction_events.location_id = locations.location_id
JOIN location_groups ON locations.location_group_id = location_groups.location_group_id
JOIN creatures ON interaction_events.creature_id = creatures.creature_id
WHERE location_groups.location_group_island_faction = '{island}'
AND interaction_events.interaction_event_time >= (SELECT MAX(interaction_event_time) - {days} FROM interaction_events)
GROUP BY creatures.creature_faction
"""

query_db(query)

,creature_faction,count
0,Growth,17
1,Stability,32
2,Light,42
3,Shadow,21


## ⏰ player events

### PLAYER_FORAGING_EVENTS

In [ ]:
query_db(f"""
SELECT
    resource_transfer_time as time,
    players.player_name as player,
    resource_types.resource_type as resource,
    resources.resource_quality as quality,
    source_collections.collection_name as source
FROM resource_transfers
JOIN
    collections as destination_collections on resource_transfers.destination_collection_id = destination_collections.collection_id
JOIN
    collections as source_collections on resource_transfers.source_collection_id = source_collections.collection_id
JOIN
    player_collection_access on destination_collections.collection_id = player_collection_access.collection_id
JOIN
    players on player_collection_access.player_id = players.player_id
JOIN
    resources on resource_transfers.resource_id = resources.resource_id
JOIN
    resource_types on resources.resource_type_id = resource_types.resource_type_id
WHERE
    destination_collections.collection_type = 'inventory'
""")

,time,player,resource,quality,source
0,0,Aisha,Shadowglass,4,Light Garden
1,0,Aisha,Crystal Water,8,Light Tree
2,0,Aisha,Sunfruit,4,Light Garden
3,0,Aisha,VegetaBulb,8,Light Tree
4,0,Aisha,Moonberry,8,Light Tree
...,...,...,...,...,...
742,9,Jasmine,Shadowberry,6,Shadow Patch
743,9,Jasmine,Shadowglass,10,Shadow Mines
744,9,Jasmine,Nightshade,6,Shadow Patch
745,9,Jasmine,Sunfruit,6,Shadow Patch


### CRAFTING EVENTS

In [ ]:
query_db(f"""
SELECT
  crafting_event_time, crafting_events.recipe_id, recipe_name, recipe_difficulty
FROM
  crafting_events
JOIN
  item_transfers ON crafting_events.item_transfer_id = item_transfers.item_transfer_id
JOIN
  items on item_transfers.item_id = items.item_id
JOIN
  recipes on items.recipe_id = recipes.recipe_id

""")

,crafting_event_time,recipe_id,recipe_name,recipe_difficulty
0,0,68,Lunar Glowberry Elixir,50
1,0,87,Crystal Water Infusion,50
2,0,14,Luminous Sunfruit Sorbet,50
3,0,48,Radiant Fig Fusion,50
4,0,67,Stardust Twilight Delight,50
...,...,...,...,...
350,9,2,Luminous Sunburst,50
351,9,46,Sunnydew Harmony,50
352,9,125,Twilight Shadow Crystal,50
353,9,88,Shadowed Nightfruit Potion,50


### PLAYER_GIFTING_EVENTS

In [ ]:
query_db(f"""
SELECT
  gifting_event_time as time,
  player_name as player,
  creatures.creature_name as creature,
  recipes.recipe_name as gift,
  items.item_id
FROM
  gifting_events
JOIN
  creatures ON gifting_events.creature_id = creatures.creature_id
JOIN
  players on gifting_events.player_id = players.player_id
JOIN
  item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
JOIN
  items on item_transfers.item_id = items.item_id
JOIN
  recipes on items.recipe_id = recipes.recipe_id
""")

,time,player,creature,gift,item_id
0,0,Aisha,Zephyrus,Crystal Water Infusion,759
1,0,Aisha,Mosswhisper,Radiant Fig Fusion,761
2,1,Aisha,Petal,Twilight Shadow Crystal,794
3,3,Aisha,Phoenix,Firmfig Ginger Delight,793
4,4,Aisha,Phoenix,Sugared Bulbs,901
...,...,...,...,...,...
154,5,Jasmine,Ironclad,Umbral Stonewood Token,827
155,7,Jasmine,Radiant,Shadowshade Night Elixir,977
156,8,Jasmine,Shadowthorn,Shadowy Mushroomburst,1046
157,9,Jasmine,Zephyrus,Umbral Stonewood Token,979


### PLAYER_CREATURE_INTERACTIONS

In [ ]:
query_db(f"""
SELECT
  interaction_event_time as time,
  location_group_name as 'map area',
  player_name as player,
  creatures.creature_name as creature,
  interaction_events.interaction_event_observed_creature_mood as 'observed mood'
FROM
  interaction_events
JOIN
  creatures ON interaction_events.creature_id = creatures.creature_id
JOIN
  players on interaction_events.player_id = players.player_id
JOIN
  locations on interaction_events.location_id = locations.location_id
JOIN
  location_groups on locations.location_group_id = location_groups.location_group_id
""")

,time,map area,player,creature,observed mood
0,3,Rock Garden,Hannah,Celestia,50
1,3,Rock Garden,Hannah,Glint,50
2,3,Rock Garden,Hannah,Ivy,50
3,3,Rock Garden,Hannah,Steelsong,48
4,4,Rock Garden,Dalia,Murk,50
...,...,...,...,...,...
327,2,Shadow Mines,Isabella,Shieldstone,50
328,2,Shadow Mines,Isabella,Lastingheart,53
329,8,Shadow Mines,Giselle,Shade,50
330,8,Shadow Mines,Giselle,Inferno,62


## 🌴 balance data

#### *All gifts given by player?*

In [ ]:
player_name = "Aisha"

result = query_db(f"""
SELECT
  gifting_event_time as time,
  creatures.creature_name as creature,
  recipes.recipe_name as gift
FROM
  gifting_events
JOIN
  creatures ON gifting_events.creature_id = creatures.creature_id
JOIN
  players on gifting_events.player_id = players.player_id
JOIN
  item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
JOIN
  items on item_transfers.item_id = items.item_id
JOIN
  recipes on items.recipe_id = recipes.recipe_id
WHERE
  players.player_name = '{player_name}'
""")

#### Faction rep by faction

In [ ]:
query_db(f"""
SELECT
    creatures.creature_faction,
    AVG(player_creature_relationships.player_creature_relationship_level) as average_relationship_level
FROM
    player_creature_relationships
JOIN
    creatures ON player_creature_relationships.creature_id = creatures.creature_id
GROUP BY
    creatures.creature_faction
ORDER BY
    average_relationship_level DESC;
""")

,creature_faction,average_relationship_level
0,Stability,51.004
1,Shadow,50.980
2,Light,50.956
3,Growth,50.932


#### All players' relationship with each faction?

In [ ]:
query_db(f"""SELECT
    players.player_name,
    creatures.creature_faction,
    AVG(player_creature_relationships.player_creature_relationship_level) as average_relationship_level
FROM
    player_creature_relationships
JOIN
    creatures ON player_creature_relationships.creature_id = creatures.creature_id
JOIN
    players ON player_creature_relationships.player_id = players.player_id
GROUP BY
    players.player_name,
    creatures.creature_faction;
""")

,player_name,creature_faction,average_relationship_level
0,Aisha,Light,51.32
1,Aisha,Shadow,50.40
2,Aisha,Growth,51.36
3,Aisha,Stability,50.88
4,Bianca,Light,50.96
5,Bianca,Shadow,50.76
6,Bianca,Growth,51.00
7,Bianca,Stability,50.80
8,Chiara,Light,50.28
9,Chiara,Shadow,50.52


# other

## **CREATURE INTERACTIONS**



#### *How many creatures were sighted in each map area each day?*

In [ ]:
query = f"""SELECT
    creature_sightings.creature_sighting_time as sighting_day,
    location_groups.location_group_name,
    COUNT(DISTINCT creature_sightings.creature_id) as creature_count
FROM creature_sightings
JOIN location_groups ON creature_sightings.location_group_id = location_groups.location_group_id
GROUP BY sighting_day, location_groups.location_group_name
ORDER BY sighting_day, location_groups.location_group_name;
"""

query_db(query)

,sighting_day,location_group_name,creature_count
0,0,Growth Patch,15
1,0,Growth Tree,6
2,0,Light Garden,14
3,0,Light Tree,12
4,0,Rock Garden,10
...,...,...,...
75,9,Light Tree,13
76,9,Rock Garden,11
77,9,Shadow Mines,12
78,9,Shadow Patch,14


#### How many of each creature faction were seen in the Light Garden on Day 0?

In [ ]:
query_db("""SELECT
    creature_sightings.creature_sighting_time as day,
    creatures.creature_faction,
    location_groups.location_group_name,
    COUNT(DISTINCT creature_sightings.creature_id) as creature_count
FROM creature_sightings
JOIN creatures ON creature_sightings.creature_id = creatures.creature_id
JOIN location_groups ON creature_sightings.location_group_id = location_groups.location_group_id
GROUP BY day, creatures.creature_faction, location_groups.location_group_name
ORDER BY day, creatures.creature_faction, location_groups.location_group_name;
""")

,day,creature_faction,location_group_name,creature_count
0,0,Growth,Growth Patch,7
1,0,Growth,Growth Tree,4
2,0,Growth,Light Tree,4
3,0,Growth,Rock Garden,2
4,0,Growth,Shadow Mines,5
...,...,...,...,...
271,9,Stability,Light Tree,6
272,9,Stability,Rock Garden,5
273,9,Stability,Shadow Mines,1
274,9,Stability,Shadow Patch,1


#### Where were the Light creatures on Day 0?

In [ ]:
faction = 'Shadow'
day = 0

query_db(f"""SELECT
    creature_sightings.creature_sighting_time as day,
    creatures.creature_faction,
    location_groups.location_group_name,
    COUNT(DISTINCT creature_sightings.creature_id) as creature_count
FROM creature_sightings
JOIN creatures ON creature_sightings.creature_id = creatures.creature_id
JOIN location_groups ON creature_sightings.location_group_id = location_groups.location_group_id
WHERE creatures.creature_faction = '{faction}'
AND creature_sightings.creature_sighting_time = '{day}'
GROUP BY day, creatures.creature_faction, location_groups.location_group_name
ORDER BY day, creatures.creature_faction, location_groups.location_group_name;
""")

,day,creature_faction,location_group_name,creature_count
0,0,Shadow,Growth Patch,3
1,0,Shadow,Light Garden,1
2,0,Shadow,Light Tree,2
3,0,Shadow,Rock Garden,1
4,0,Shadow,Shadow Mines,9
5,0,Shadow,Shadow Patch,6
6,0,Shadow,Stability Mines,3


## **PLAYERS AND INVENTORIES**

#### *What resources does Aisha have in her inventory?*

In [ ]:
player_name = "Aisha"

query_db(f"""
SELECT
    resource_types.resource_type,
    COUNT(resources.resource_id) as resource_count
FROM
    resources
JOIN
    collections ON resources.collection_id = collections.collection_id
JOIN
    player_collection_access ON collections.collection_id = player_collection_access.collection_id
JOIN
    players ON player_collection_access.player_id = players.player_id
JOIN
    resource_types ON resources.resource_type_id = resource_types.resource_type_id
WHERE
    collections.collection_type = 'inventory' AND players.player_name = '{player_name}'
GROUP BY
    resource_types.resource_type;
""")

NameError: name 'query_db' is not defined

#### *What items does Aisha have in her inventory?*

In [ ]:
player_name = "Aisha"

query_db(f"""
SELECT
    recipes.recipe_name,
    COUNT(items.item_id) as item_count
FROM
    items
JOIN
    collections ON items.collection_id = collections.collection_id
JOIN
    player_collection_access ON collections.collection_id = player_collection_access.collection_id
JOIN
    players ON player_collection_access.player_id = players.player_id
JOIN
    recipes ON items.recipe_id = recipes.recipe_id
WHERE
    collections.collection_type = 'inventory' AND players.player_name = 'Aisha'
GROUP BY
    recipes.recipe_name;
""")

,recipe_name,item_count
0,Stardust Twilight Delight,1


#### *How many items does Aisha have from each faction?*

In [ ]:
player_name = "Aisha"

query_db(f"""
SELECT
    resource_types.resource_type_faction,
    COUNT(*) as count
FROM
    resources
JOIN
    resource_types ON resources.resource_type_id = resource_types.resource_type_id
JOIN
    collections ON resources.collection_id = collections.collection_id
JOIN
    player_collection_access on collections.collection_id = player_collection_access.collection_id
JOIN
    players on player_collection_access.player_id = players.player_id
WHERE
    players.player_name = '{player_name}'
GROUP BY resource_types.resource_type_faction;
""")

,resource_type_faction,count
0,Shadow,2
1,Light,2
2,Growth,1


#### *How many of each item does each player have?*

In [ ]:
query_db(f"""
SELECT
    recipes.recipe_name as item,
    collections.collection_name as inventory,
    COUNT(*) as count
FROM
    items
JOIN
    recipes ON items.recipe_id = recipes.recipe_id
JOIN
    collections ON items.collection_id = collections.collection_id
WHERE
    collections.collection_type = 'inventory'
GROUP BY recipes.recipe_name, collections.collection_name;
""")

,item,inventory,count
0,Lunar Glowberry Elixir,Aisha's Inventory,1
1,Stellar Sweetwater,Aisha's Inventory,3
2,Lunar Moonberry Potion,Aisha's Inventory,1
3,Crystal Sweetfire Potion,Aisha's Inventory,1
4,Shadowed Sweetberry,Aisha's Inventory,1
...,...,...,...
153,Shadowed Crystal Potion,Jasmine's Inventory,1
154,Serene Mossy Concoction,Jasmine's Inventory,1
155,Shadowshade Night Elixir,Jasmine's Inventory,1
156,Umbral Nightshade Potion,Jasmine's Inventory,1


In [ ]:
# Execute the query
results = query_db(f"""
SELECT collections.collection_name, resource_types.resource_type, COUNT(*) as count
FROM resources
JOIN resource_types ON resources.resource_type_id = resource_types.resource_type_id
JOIN collections ON resources.collection_id = collections.collection_id

GROUP BY Collections.name, Resources.type;
""")

# Convert the DataFrame to a pivot table
pivot_table = results.pivot(index='name', columns='type', values='count')

# Fill NaN with 0
pivot_table.fillna(0, inplace=True)

print(pivot_table)

ProgrammingError: ignored

#### *Who has the most magic items?*

In [ ]:
resource_type_category = "Food"

query_db(f"""\
SELECT
    players.player_name as player,
    COUNT(resource_types.resource_type_category) as '{resource_type_category} items'
FROM resources
JOIN resource_types ON resources.resource_type_id = resource_types.resource_type_id
JOIN collections ON resources.collection_id = collections.collection_id
JOIN player_collection_access ON collections.collection_id = player_collection_access.collection_id
JOIN players ON player_collection_access.player_id = players.player_id
WHERE resource_types.resource_type_category = '{resource_type_category}' AND collections.collection_type = 'inventory'
GROUP BY players.player_name
ORDER BY '{resource_type_category} items' DESC;
""")

## **FORAGING**

In [ ]:
query_db("""
SELECT DISTINCT `c1`.`collection_name` AS `source`, `c2`.`collection_name` AS `destination_collection_name`
FROM `resource_transfers` AS `rte`
JOIN `collections` AS `c1` ON `rte`.`source_collection_id` = `c1`.`collection_id`
JOIN `collections` AS `c2` ON `rte`.`destination_collection_id` = `c2`.`collection_id`
JOIN `player_collection_access` AS `ca` ON `ca`.`collection_id` = `c2`.`collection_id`
JOIN `players` AS `p` ON `ca`.`player_id` = `p`.`player_id`
WHERE `p`.`player_name` = 'Aisha'
AND `c2`.`collection_type` = 'inventory'
""")

,source,destination_collection_name
0,Light Garden,Aisha's Inventory
1,Light Tree,Aisha's Inventory


### *Where did Aisha get her resources?*

#### *What are the names and types of all the creatures?*

In [ ]:
query_db("""
SELECT
    creature_name,
    creature_faction
FROM
    creatures
""")

,creature_name,creature_faction
0,Luminara,Light
1,Prismaros,Light
2,Aurora,Light
3,Celestia,Light
4,Radiant,Light
...,...,...
95,Aurum,Stability
96,Guardian,Stability
97,Ironcliff,Stability
98,Steelsong,Stability


#### *What has Aisha crafted?*

In [ ]:
player_name= "Aisha"

query_db(f"""\
SELECT
    recipes.recipe_name
FROM
    items
JOIN
    recipes ON items.recipe_id = recipes.recipe_id
JOIN
    collections ON items.collection_id = collections.collection_id
JOIN
    player_collection_access on collections.collection_id = player_collection_access.collection_id
JOIN
    players on player_collection_access.player_id = players.player_id
WHERE
    players.player_name = '{player_name}';
""")

,recipe_name
0,Lunar Glowberry Elixir
1,Stellar Sweetwater
2,Lunar Moonberry Potion
3,Crystal Sweetfire Potion
4,Shadowed Sweetberry
5,Stardust Glitter Delight
6,Stellar Sweetwater
7,Harvest Cake
8,Stellar Sweetwater
9,Luminous Sunburst


## **CRAFTING**

### recipes

#### *List all recipes?*

In [ ]:
query_db("SELECT * from recipes")

,recipe_id,recipe_name,recipe_category,recipe_temp,recipe_difficulty
0,1,Rusted Stardusties,Food,140,50
1,2,Luminous Sunburst,Food,140,50
2,3,Stardust Glitter Delight,Food,140,50
3,4,Glowing Shadowsoup,Food,140,50
4,5,Shadowed Storm Soup,Food,140,50
...,...,...,...,...,...
137,138,Stonewater Essence,Gift,140,50
138,139,Serene Hardwood Essence,Gift,140,50
139,140,ShaToken,Gift,140,50
140,141,Crystal Water Reflection,Gift,140,50


#### *Ingredients in a specific recipe?*

In [ ]:
recipe_name = "Ironroot Glowstew"

query_db(f"SELECT recipe_name, resource_type \
FROM recipes \
JOIN recipe_ingredient_resource_types ON recipes.recipe_id = recipe_ingredient_resource_types.recipe_id \
JOIN resource_types ON recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id \
WHERE recipes.recipe_name = '{recipe_name}';")

,recipe_name,resource_type
0,Ironroot Glowstew,Glowbug
1,Ironroot Glowstew,Ironroot


#### *All recipes that use a specific ingredient?*

In [ ]:
ingredient_name = 'Glowbug'
query_db(f"SELECT recipes.* \
FROM resource_types \
JOIN recipe_ingredient_resource_types ON resource_types.resource_type_id = recipe_ingredient_resource_types.resource_type_id \
JOIN recipes ON recipe_ingredient_resource_types.recipe_id = recipes.recipe_id \
WHERE resource_types.resource_type = '{ingredient_name}';")

,recipe_id,recipe_name,recipe_category,recipe_temp,recipe_difficulty
0,2,Luminous Sunburst,Food,140,50
1,3,Stardust Glitter Delight,Food,140,50
2,4,Glowing Shadowsoup,Food,140,50
3,5,Shadowed Storm Soup,Food,140,50
4,6,Shadowberry Dream,Food,140,50
5,7,Luminous Veggie Delight,Food,140,50
6,8,Nectarous Glowdew,Food,140,50
7,9,Luminous Sugarleaf Tart,Food,140,50
8,10,Ironroot Glowstew,Food,140,50
9,11,Stalwart Fig Delight,Food,140,50


#### *How many recipes can be made with each ingredient?*

In [ ]:
query_db(f"""
SELECT
    resource_types.resource_type as ingredient,
    COUNT(recipes.recipe_id) as num_recipes
FROM
    resource_types
JOIN
    recipe_ingredient_resource_types ON resource_types.resource_type_id = recipe_ingredient_resource_types.resource_type_id
JOIN
    recipes ON recipe_ingredient_resource_types.recipe_id = recipes.recipe_id
GROUP BY
    resource_types.resource_type
ORDER BY num_recipes DESC;
""")

,ingredient,num_recipes
0,Moonberry,19
1,Nightshade,19
2,Tranquil Moss,19
3,Crystal Water,19
4,Glowbug,15
5,Sunfruit,15
6,Sugary Stardust,15
7,Ghostcap Mushroom,15
8,Stormfruit,15
9,Shadowberry,15


#### *Get all ingredients needed for a list of recipes?*

In [ ]:
recipe_names = ['Ironroot Glowstew', 'Tranquility Moss Tonic']
recipe_names_str = ', '.join(f"'{name}'" for name in recipe_names)

query_db(f"""
SELECT
    recipes.recipe_name,
    resource_types.resource_type as ingredient
FROM
    recipes
JOIN
    recipe_ingredient_resource_types ON recipes.recipe_id = recipe_ingredient_resource_types.recipe_id
JOIN
    resource_types ON recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id
WHERE
    recipes.recipe_name IN ({recipe_names_str});
""")

,recipe_name,ingredient
0,Ironroot Glowstew,Glowbug
1,Ironroot Glowstew,Ironroot
2,Tranquility Moss Tonic,Glowbug
3,Tranquility Moss Tonic,Tranquil Moss


### crafting activity

#### *How many of each item type have been crafted?*

In [ ]:
query_db("""
SELECT recipes.recipe_name, COUNT(item_transfers.item_id) as item_count
FROM item_transfers
JOIN items ON item_transfers.item_id = items.item_id
JOIN recipes ON items.recipe_id = recipes.recipe_id
WHERE item_transfers.source_collection_id IN
    (SELECT collection_id FROM collections WHERE collection_type = 'crafting_table')
GROUP BY recipes.recipe_name
ORDER BY item_count DESC;
""")

,recipe_name,item_count
0,Twilight Stormfruit Surprise,3
1,Firmfig Ginger Delight,2
2,Lunar Moonberry Potion,2
3,ShaToken,2
4,Rusted Stardusties,1
5,Glowing Shadowsoup,1
6,Nectarous Glowdew,1
7,Umbral Shadowberry Bliss,1
8,Glowing Sugarleaf Confection,1
9,Stardusted Figgies,1


## **CREATURE RELATIONSHIPS**

### *What's Aisha's friendship level with Aurora?*

In [ ]:
player_name = "Aisha"
creature_name = "Aurora"

results = query_db(f"\
SELECT player_creature_relationships.player_creature_relationship_level \
FROM players \
JOIN player_creature_relationships ON players.player_id = player_creature_relationships.player_id \
JOIN creatures ON player_creature_relationships.creature_id = creatures.creature_id \
WHERE players.player_name = '{player_name}' AND creatures.creature_name = '{creature_name}';")

results

,player_creature_relationship_level
0,51


#### FRIENDSHIPS

In [ ]:
threshold = 6

query_db(f"""
SELECT
    players.player_name,
    creatures.creature_name,
    player_creature_relationships.player_creature_relationship_level
FROM
    player_creature_relationships
JOIN
    creatures ON player_creature_relationships.creature_id = creatures.creature_id
JOIN
    players ON player_creature_relationships.player_id = players.player_id
WHERE
    player_creature_relationships.player_creature_relationship_level > {threshold};

""")

,player_name,creature_name,player_creature_relationship_level
0,Aisha,Luminara,50
1,Aisha,Prismaros,50
2,Aisha,Aurora,51
3,Aisha,Celestia,50
4,Aisha,Radiant,50
...,...,...,...
995,Jasmine,Aurum,51
996,Jasmine,Guardian,50
997,Jasmine,Ironcliff,50
998,Jasmine,Steelsong,55


#### Friendliest players?

In [ ]:
query_db(f"""SELECT
    players.player_name,
    AVG(player_creature_relationships.player_creature_relationship_level) as avg_relationship_level
FROM
    player_creature_relationships
JOIN
    players ON player_creature_relationships.player_id = players.player_id
GROUP BY
    players.player_name
ORDER BY
    avg_relationship_level DESC
LIMIT 5;
""")

,player_name,avg_relationship_level
0,Giselle,51.38
1,Dalia,51.12
2,Jasmine,51.10
3,Emiko,51.04
4,Aisha,50.99


## **GIFTING**

### *How many gifts have each faction received?*

In [ ]:
query_db("""SELECT creatures.creature_faction, COUNT(item_id) as gifts_received
FROM gifting_events
JOIN creatures ON gifting_events.creature_id = creatures.creature_id
JOIN item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
GROUP BY creatures.creature_faction
ORDER BY gifts_received DESC
""")

,creature_faction,gifts_received
0,Shadow,41
1,Stability,40
2,Growth,39
3,Light,39


### *Do gifts make creatures healthier?*

In [ ]:
query_db("""
SELECT
  creatures.creature_name as creature,
  COUNT(*) as gifts_received,
  AVG(creatures.creature_health) as health
FROM
  gifting_events
JOIN
  creatures ON gifting_events.creature_id = creatures.creature_id
LEFT JOIN
  interaction_events on gifting_events.gifting_event_id = interaction_events.gifting_event_id
JOIN
  item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
WHERE
  creatures.creature_health < '100'
GROUP BY creatures.creature_name
ORDER BY count(*) DESC;
""")

,creature,gifts_received,health
0,Solidarity,4,75.0
1,Nocturna,4,79.0
2,Phoenix,4,89.0
3,Lastingheart,3,80.0
4,Firmament,3,72.0
...,...,...,...
66,Strongroot,1,52.0
67,Solara,1,51.0
68,Aurum,1,77.0
69,Ironcliff,1,62.0


### *Do gifts make creatures happier?*

In [ ]:
query_db("""
SELECT
    creatures.creature_name as creature,
    COUNT(gifting_events.item_transfer_id) as gifts_received,
    creatures.creature_mood as mood
FROM
    creatures
LEFT JOIN
    gifting_events ON gifting_events.creature_id = creatures.creature_id
GROUP BY
    creatures.creature_id
ORDER BY creature_mood DESC;
""")

,creature,gifts_received,mood
0,Gloomsong,2,140
1,Radiance,4,137
2,Flora,3,119
3,Lastingheart,3,107
4,Petal,2,107
...,...,...,...
95,Fern,1,38
96,Nocturna,4,36
97,Shadowthorn,2,36
98,Zephyrus,2,32


### *Have any items been gifted a lot recently?*

In [ ]:
start_date = 0
end_date = 10

query_db(f"""
SELECT recipes.recipe_name, COUNT(item_transfers.item_id) as times_gifted
FROM gifting_events
JOIN item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
JOIN items on item_transfers.item_id = items.item_id
JOIN recipes on items.recipe_id = recipes.recipe_id
WHERE gifting_events.gifting_event_time BETWEEN '{start_date}' AND '{end_date}'
GROUP BY recipes.recipe_name
ORDER BY times_gifted DESC;
""")

,recipe_name,times_gifted
0,Twilight Shadow Crystal,10
1,Serene Hardwood Essence,7
2,Luminous Sunburst,5
3,Umbral Stonewood Token,5
4,Stardust Twilight Delight,4
...,...,...
70,Illuminated Crystal Elixir,1
71,Rusted Stardusties,1
72,Growthfire Leaf Fusion,1
73,Lunar Glowroot Concoction,1


#### *Who has Aisha interacted with, and where?*

In [ ]:
player_name = "Aisha"

query_db(f"""
SELECT interaction_event_time as 'day', creature_name as 'creature', location_group_name as 'area'
FROM interaction_events
JOIN players ON interaction_events.player_id = players.player_id
JOIN creatures on interaction_events.creature_id = creatures.creature_id
JOIN locations on interaction_events.location_id = locations.location_id
JOIN location_groups on locations.location_group_id = location_groups.location_group_id
WHERE players.player_name = '{player_name}';
""")

,day,creature,area
0,0,Blaze,Light Tree
1,0,Viridium,Light Tree
2,0,Zephyrus,Light Tree
3,0,Mosswhisper,Light Tree
4,0,Gleam,Light Tree
5,1,Petal,Light Garden
6,1,Aurum,Light Garden
7,2,Lumigrove,Growth Patch
8,2,Rose,Growth Patch
9,3,Aurora,Light Tree


In [ ]:
query_db("""
SELECT creatures.creature_name, COUNT(item_id) as items_gifted
FROM item_transfers
JOIN creatures ON item_transfers.destination_collection_id = creatures.collection_id
WHERE item_transfers.item_transfer_time BETWEEN '0' AND '10'
GROUP BY creatures.creature_name
ORDER BY items_gifted DESC;
""")

,creature_name,items_gifted
0,Nocturna,4
1,Aurelia,4
2,Phoenix,4
3,Solidarity,4
4,Vanguard,4
...,...,...
77,Solara,1
78,Strongroot,1
79,Blossom,1
80,Vivaro,1


### *How many gifts has each creature received?*

#### *What gifts has Aisha given?*

In [ ]:
player_name = "Aisha"

query_db(f"""
SELECT gifting_event_time as 'day', player_name as 'player', creature_name as 'creature', recipes.recipe_name as 'recipe'
FROM gifting_events
JOIN players ON gifting_events.player_id = players.player_id
JOIN creatures on gifting_events.creature_id = creatures.creature_id
JOIN item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
JOIN items ON item_transfers.item_id = items.item_id
JOIN recipes on items.recipe_id = recipes.recipe_id
WHERE players.player_name = '{player_name}'
""")

,day,player,creature,recipe
0,0,Aisha,Rose,Shadowberry Dream
1,0,Aisha,Gleam,Tranquil Stardust Elixir
2,0,Aisha,Radiant Blossom,Glowing Shadowsoup
3,2,Aisha,Endurance,Lunar Glowberry Elixir
4,3,Aisha,Nightshade,Enchanted Hardwood Charm
5,4,Aisha,Rose,Stardust Glitter Delight
6,5,Aisha,Prismaros,Crystal Water Infusion
7,5,Aisha,Gleam,Umbral Shadowberry Bliss
8,7,Aisha,Luminary,Stardust Twilight Delight
9,7,Aisha,Shade,Crystal Sunwater Elixir


#### *How many creatures were interacted with in the Rock Garden each day?*

In [ ]:
area_name = "Rock Garden"

query = f"""
SELECT
    interaction_events.interaction_event_time as day,
    COUNT(DISTINCT interaction_events.creature_id) AS interactions
FROM interaction_events
JOIN locations ON interaction_events.location_id = locations.location_id
JOIN location_groups ON locations.location_group_id = location_groups.location_group_id
WHERE location_groups.location_group_name = '{area_name}'
GROUP BY day;
"""

query_db(query)

,day,interactions
0,1,8
1,2,2
2,4,4
3,6,3
4,7,3
5,8,4


#### *How many creatures were sighted in the Rock Garden each day?*

In [ ]:
area_name = "Rock Garden"

query = f"""
WITH
rock_garden_sightings AS (
    SELECT
        creature_sightings.creature_sighting_time as sighting_day,
        COUNT(DISTINCT creature_sightings.creature_id) AS distinct_creature_count
    FROM creature_sightings
    JOIN location_groups ON creature_sightings.location_group_id = location_groups.location_group_id
    WHERE location_groups.location_group_name = '{area_name}'
    GROUP BY sighting_day
    ORDER BY distinct_creature_count DESC
),
light_garden_sightings AS (
    SELECT
        creature_sightings.creature_sighting_time as sighting_day,
        COUNT(DISTINCT creature_sightings.creature_id) AS distinct_creature_count
    FROM creature_sightings
    JOIN location_groups ON creature_sightings.location_group_id = location_groups.location_group_id
    WHERE location_groups.location_group_name = 'Light Garden'
    GROUP BY sighting_day
    ORDER BY distinct_creature_count DESC
)

SELECT
    rgs.sighting_day,
    rgs.distinct_creature_count as rock_garden_sightings,
    lgs.distinct_creature_count as light_garden_sightings,
    CASE
        WHEN rgs.distinct_creature_count > lgs.distinct_creature_count THEN 'more rock creatures'
        WHEN lgs.distinct_creature_count > rgs.distinct_creature_count THEN 'more light creatures'
        ELSE 'same'
    END AS comparison,
    MAX(rgs.distinct_creature_count) OVER () as max_rock_creatures,
    RANK() OVER(ORDER BY rgs.distinct_creature_count)
FROM
    rock_garden_sightings rgs
    LEFT JOIN light_garden_sightings lgs ON rgs.sighting_day = lgs.sighting_day
;
"""

query_db(query)

ProgrammingError: ignored

#### all sightings by faction

In [ ]:
query_db(f"""SELECT
    DATE(creature_sightings.creature_sighting_time) as sighting_day,
    creatures.creature_faction,
    COUNT(DISTINCT creature_sightings.creature_id) as creature_count
FROM creature_sightings
JOIN creatures ON creature_sightings.creature_id = creatures.creature_id
GROUP BY sighting_day, creatures.creature_faction
ORDER BY sighting_day, creatures.creature_faction;
""")

,sighting_day,creature_faction,creature_count
0,None,Growth,25
1,None,Light,25
2,None,Shadow,25
3,None,Stability,25


#### *Which creature has received the most gifts?*

In [ ]:
query_db("""
SELECT creatures.creature_name, COUNT(item_id) as gifts_received
FROM gifting_events
JOIN creatures ON gifting_events.creature_id = creatures.creature_id
JOIN item_transfers ON gifting_events.item_transfer_id = item_transfers.item_transfer_id
GROUP BY creatures.creature_name
ORDER BY gifts_received DESC
LIMIT 5;
""")

,creature_name,gifts_received
0,Vanguard,4
1,Phoenix,4
2,Nocturna,4
3,Radiance,4
4,Aurelia,4


## TABLE VIEWS

### TABLE DESCRIPTIONS

In [ ]:
query_db(f"""
    SHOW TABLES;
""")

,Tables_in_gameplay-data
0,collections
1,crafting_events
2,creatures
3,gifting_events
4,interaction_events
5,item_transfers
6,items
7,location_groups
8,locations
9,player_collection_access


### PLAYER_ACCESSIBLE_INVENTORIES VIEW

In [ ]:
query_db(f"""
SELECT
    players.player_id,
    players.player_name,
    collections.collection_id,
    collections.collection_name,
    player_collection_access_level
FROM collections
JOIN
    player_collection_access on collections.collection_id = player_collection_access.collection_id
JOIN
    players on player_collection_access.player_id = players.player_id
WHERE
    collection_type = 'inventory'
""")

,player_id,player_name,collection_id,collection_name,player_collection_access_level
0,1,Aisha,111,Aisha's Inventory,1
1,2,Bianca,114,Bianca's Inventory,1
2,3,Chiara,117,Chiara's Inventory,1
3,4,Dalia,120,Dalia's Inventory,1
4,5,Emiko,123,Emiko's Inventory,1
5,6,Fatima,126,Fatima's Inventory,1
6,7,Giselle,129,Giselle's Inventory,1
7,8,Hannah,132,Hannah's Inventory,1
8,9,Isabella,135,Isabella's Inventory,1
9,10,Jasmine,138,Jasmine's Inventory,1


### PLAYER_ACCESSIBLE_CRAFTING_TABLES

In [ ]:
query_db(f"""
SELECT
    players.player_id,
    players.player_name,
    collections.collection_id,
    collections.collection_name,
    player_collection_access_level
FROM collections
JOIN
    player_collection_access on collections.collection_id = player_collection_access.collection_id
JOIN
    players on player_collection_access.player_id = players.player_id
WHERE
    collection_type = 'crafting_table'
""")

,player_id,player_name,collection_id,collection_name,player_collection_access_level
0,1,Aisha,112,Aisha's Crafting Table,1
1,2,Bianca,115,Bianca's Crafting Table,1
2,3,Chiara,118,Chiara's Crafting Table,1
3,4,Dalia,121,Dalia's Crafting Table,1
4,5,Emiko,124,Emiko's Crafting Table,1
5,6,Fatima,127,Fatima's Crafting Table,1
6,7,Giselle,130,Giselle's Crafting Table,1
7,8,Hannah,133,Hannah's Crafting Table,1
8,9,Isabella,136,Isabella's Crafting Table,1
9,10,Jasmine,139,Jasmine's Crafting Table,1


### RECIPE_INGREDIENTS

In [ ]:
query_db(f"""
SELECT
  recipes.recipe_id,
  recipe_name,
  recipe_category,
  resource_types.resource_type_id,
  resource_type,
  resource_type_faction,
  resource_type_rarity
FROM
  recipe_ingredient_resource_types
JOIN
  recipes ON recipes.recipe_id = recipe_ingredient_resource_types.recipe_id
JOIN
  resource_types on recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id
""")

,recipe_id,recipe_name,recipe_category,resource_type_id,resource_type,resource_type_faction,resource_type_rarity
0,2,Luminous Sunburst,Food,1,Glowbug,Light,4
1,3,Stardust Glitter Delight,Food,1,Glowbug,Light,4
2,4,Glowing Shadowsoup,Food,1,Glowbug,Light,4
3,5,Shadowed Storm Soup,Food,1,Glowbug,Light,4
4,6,Shadowberry Dream,Food,1,Glowbug,Light,4
...,...,...,...,...,...,...,...
278,132,Shadowstone Enigma,Gift,20,Stonewood,Stability,3
279,134,Stalwart Stonewood Tranquility,Gift,20,Stonewood,Stability,3
280,137,Umbral Stonewood Token,Gift,20,Stonewood,Stability,3
281,138,Stonewater Essence,Gift,20,Stonewood,Stability,3


# Data Representations

In [ ]:
show_as_table("Amethyst 's Location Memories")

In [ ]:
show_as_bar_chart("The Health Oracle", 'creature_faction', 'creature_health')

In [ ]:
s = squish("The Health Oracle", "Creature Faction Lookup Book")

s2 = squish(s, "The Mood Oracle")

show_as_bar_chart(s2, 'creature_faction', 'creature_mood')

In [ ]:
title = group_averager("The Health Oracle", ['creature_faction'], 'creature_health')

show_as_bar_chart(title, 'creature_faction', 'AVG(creature_health)')

KeyError: ignored

In [ ]:
show_as_plot("Amethyst 's Location Memories", 'location_group_name', 'player_name')

In [ ]:
show_as_plot("The Health Oracle", 'creature_faction', 'creature_health')

In [ ]:
show_as_plot("The Mood Oracle", 'creature_faction', 'creature_mood')

In [ ]:
show_as_plot("The Health Oracle", 'creature_faction', 'creature_health')

In [ ]:
#show("Aurora's Location Memories")
show_as_map(title="Aurora's Location Memories", x='location_x', y='location_y', color_column=None)

KeyError: ignored